# Embedding Generation and Qdrant Storage

This notebook converts the retrieval corpus to embeddings using BAAI/bge-large-en-v1.5 and stores them in Qdrant with repo-aware filtering capabilities.


In [1]:
import json
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Load retrieval corpus
with open('../data/processed/retrival_corpus.json', 'r') as f:
    corpus = json.load(f)

print(f"Loaded {len(corpus)} chunks from retrieval corpus")

c:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 505 chunks from retrieval corpus


In [2]:
corpus[0]

{'text': 'Use 4 spaces per indentation level.',
 'category': 'indentation',
 'source_type': 'pep8',
 'source_path': 'https://peps.python.org/pep-0008/#indentation',
 'chunk_id': 'chunk_0001'}

In [3]:
# Generate embeddings using BAAI/bge-large-en-v1.5
model = SentenceTransformer('BAAI/bge-large-en-v1.5')

texts = [chunk['text'] for chunk in corpus]
embeddings = model.encode(texts, show_progress_bar=True)

print(f"Generated embeddings with shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2535.62it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 16/16 [03:13<00:00, 12.08s/it]

Generated embeddings with shape: (505, 1024)
Embedding dimension: 1024


In [4]:
# Initialize Qdrant client (in-memory for development)
client = QdrantClient(url="http://localhost:6333")

collection_name = "guideline_embeddings"
vector_size = embeddings.shape[1]

# Create collection with vector configuration
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
)

print(f"Created Qdrant collection: {collection_name}")

Created Qdrant collection: guideline_embeddings


In [5]:
# Upload embeddings to Qdrant with metadata for filtering
points = []
for idx, chunk in enumerate(corpus):
    point = PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload={
            'text': chunk['text'],
            'category': chunk['category'],
            'source_type': chunk['source_type'],
            'chunk_id': chunk['chunk_id']
        }
    )
    points.append(point)

client.upsert(
    collection_name=collection_name,
    points=points
)

print(f"Uploaded {len(points)} embeddings to Qdrant")

Uploaded 505 embeddings to Qdrant


In [12]:
# all unique source_type in the corpus
source_types = set(chunk['source_type'] for chunk in corpus)
print("Unique source types in corpus:")
for st in source_types:
    print(f"- {st}")

Unique source types in corpus:
- pep8
- flask_guidelines
- flask_review_comment
- sklearn_guidelines
- ruff
- django_guidelines
- pandas_guidelines
- pep257
- django_review_comment
- sklearn_review_comment
- flake8
- pandas_review_comment
- pylint
- fastapi_review_comment


In [8]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

# Query function with repo-aware filtering
def query_guidelines(query_text, repo_name=None, limit=5):
    """
    Query guidelines with optional repo filtering.

    Args:
        query_text: Natural language query (or summarized query from code diff)
        repo_name: Optional repo name (e.g., 'django', 'flask'). If None, returns common guidelines only
        limit: Number of results to return
    """
    query_embedding = model.encode(query_text).tolist()

    # Build filter conditions for source_type
    common_sources = ["pep8", "flake8", "pylint", "ruff", "pep257"]
    should_conditions = [
        FieldCondition(key="source_type", match=MatchValue(value=source))
        for source in common_sources
    ]

    if repo_name:
        # Match either repo-specific guidelines OR common guidelines
        should_conditions.insert(
            0,
            FieldCondition(
            key="source_type",
            match=MatchValue(value=f"{repo_name}_guidelines"),
            ),
        )
        should_conditions.insert(
            1,
            FieldCondition(
            key="source_type",
            match=MatchValue(value=f"{repo_name}_review_comment"),
            ),
        )

    response = client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        query_filter=Filter(should=should_conditions),
        limit=limit,
    )

    return response.points

print("Query function defined")

Query function defined


In [9]:
# Intelligent query generation from code diff
def generate_query_from_diff(code_diff, diff_context):
    """
    Convert a code diff to an intelligent query by extracting context.
    
    Args:
        code_diff: The actual code diff/changes
        diff_context: Context about what was changed (e.g., "indentation issue", "naming convention", "import organization")
    
    Returns:
        Meaningful query string for embedding lookup
    """
    # Extract key terms from diff
    lines = code_diff.split('\n')
    added_lines = [l[1:].strip() for l in lines if l.startswith('+') and not l.startswith('+++')][:3]
    removed_lines = [l[1:].strip() for l in lines if l.startswith('-') and not l.startswith('---')][:3]
    
    # Combine context with extracted terms
    query = f"{diff_context} code changes: {' '.join(removed_lines + added_lines)}"
    return query

# Example: Intelligent query for django repo with code violation
example_diff = """
-def MyFunction(arg):
+def my_function(arg):
     pass
"""

example_query = generate_query_from_diff(example_diff, "function naming convention")
print(f"Generated Query: {example_query}")

# Search with intelligent query
results = query_guidelines(example_query, repo_name='django', limit=3)
print(f"\nFound {len(results)} relevant guidelines for django")
for hit in results:
    print(f"- Score: {hit.score:.4f}")
    print(f"  Type: {hit.payload['source_type']}")
    print(f"  Category: {hit.payload['category']}")
    print(f"  Text: {hit.payload['text'][:80]}...\n")

Generated Query: function naming convention code changes: def MyFunction(arg): def my_function(arg):

Found 3 relevant guidelines for django
- Score: 0.7424
  Type: django_guidelines
  Category: naming_convention
  Text: Use underscores, not camelCase, for variable, function and method names....

- Score: 0.7385
  Type: ruff
  Category: naming_convention
  Text: N807: Function name should not start and end with double underscores (__). Dunde...

- Score: 0.7339
  Type: ruff
  Category: naming_convention
  Text: N802: Function name should be lowercase. Use snake_case for function names....



In [10]:
# Example 1: Query for Django-specific guidelines
print("=" * 60)
print("Example 1: Django-specific query on indentation")
print("=" * 60)
results = query_guidelines("proper indentation with 4 spaces", repo_name='django', limit=3)
for hit in results:
    print(f"Source: {hit.payload['source_type']} | Score: {hit.score:.4f}")
    print(f"Text: {hit.payload['text']}\n")

# Example 2: Query for Flask-specific guidelines
print("\n" + "=" * 60)
print("Example 2: Flask-specific query on naming conventions")
print("=" * 60)
results = query_guidelines("function naming should be lowercase snake_case", repo_name='flask', limit=3)
for hit in results:
    print(f"Source: {hit.payload['source_type']} | Score: {hit.score:.4f}")
    print(f"Text: {hit.payload['text']}\n")

# Example 3: Query for common guidelines only (no repo filter)
print("\n" + "=" * 60)
print("Example 3: Common guidelines (PEP8, flake8, pylint, ruff)")
print("=" * 60)
results = query_guidelines("line length and formatting", repo_name=None, limit=3)
for hit in results:
    print(f"Source: {hit.payload['source_type']} | Score: {hit.score:.4f}")
    print(f"Text: {hit.payload['text']}\n")

Example 1: Django-specific query on indentation
Source: pep8 | Score: 0.8991
Text: Use 4 spaces per indentation level.

Source: flake8 | Score: 0.8069
Text: E111: Indentation is not a multiple of four spaces.

Source: ruff | Score: 0.8051
Text: Ruff E111: Indentation is not a multiple of the configured indent-width (default 4). Per PEP 8, use 4 spaces per indentation level.


Example 2: Flask-specific query on naming conventions
Source: ruff | Score: 0.9357
Text: N802: Function name should be lowercase. Use snake_case for function names.

Source: ruff | Score: 0.9130
Text: N803: Argument name should be lowercase. Use snake_case for function arguments.

Source: ruff | Score: 0.8178
Text: N816: Variable in global scope should not use mixedCase. Use snake_case instead.


Example 3: Common guidelines (PEP8, flake8, pylint, ruff)
Source: ruff | Score: 0.7002
Text: D200: One-line docstring should fit on one line. Do not spread a short docstring across multiple lines.

Source: flake8 | Score: